# Kombination der Experten: Gleichgewicht, statische Gewichte, Gating

Dieses Notebook baut auf `volatilitaet_chronos2_har_garch.ipynb` auf und liest **nur die dort gespeicherten Forecasts** aus `results/`. Es wird nichts neu prognostiziert. Neue Experten erscheinen hier automatisch, sobald ihre Parquet-Dateien existieren und sie in `POOL` stehen.

**Kombinationsstrategien** (alle kombinieren Mittelwert *und* Quantile mit demselben Gewichtsvektor $w \in \Delta^{M-1}$):

| Methode | Gewichte | gelernt auf |
|---|---|---|
| Gleichgewicht | $w_m = 1/M$ | – |
| Auswahl | One-Hot auf das beste Einzelmodell in `train` | train |
| Statisch | ein Gewichtsvektor pro Horizont, $w = \mathrm{softmax}(b)$ | train |
| Gating linear | $w(x_t) = \mathrm{softmax}(b + W x_t)$ pro Zeitreihe und Ursprung (Shazeer et al. 2017, ohne Sparsity) | train, Regularisierung auf val |
| Gating MLP | wie oben, mit einer versteckten Schicht | train, Regularisierung auf val |

Kombinierter Mittelwert $\hat\mu = \sum_m w_m \hat\mu_m$, kombinierte Quantile $\hat q_\tau = \sum_m w_m \hat q_{m,\tau}$ (Quantil-Mittelung, bleibt monoton).

**Trainingsziel:** $\text{QLIKE}(\hat\mu) + \alpha \cdot \text{Pinball}(\hat q)/s_t$. Der Pinball-Loss wird durch das RV-Niveau $s_t$ der letzten 22 Tage geteilt, damit Reihen mit hoher Vola das Training nicht dominieren.

## Splits und faire Zero-Shot-Evaluation

Ein **Split** teilt die Forecasts nach ihrem *Zieldatum* in drei Perioden:

- **train**: Hier lernen die Kombinationen ihre Gewichte.
- **val**: Hier werden Hyperparameter gewählt (z.B. Regularisierung des Gates), und wir schauen uns Ergebnisse an.
- **test**: Gesperrt bis zur finalen Auswertung. Wird nur einmal geöffnet.

**Warum die Grenzen verschoben wurden:** Die Foundation Models wurden auf öffentlichen Daten vortrainiert, möglicherweise auch auf Krypto-Kursen. Liegt eine Evaluationsperiode *vor* dem Trainings-Cutoff eines Modells, kann es die Werte schon gesehen haben, und der Vergleich wäre nicht mehr zero-shot. Der Cutoff ist meist nicht dokumentiert, das Release-Datum ist eine obere Schranke:

| Experte | Release (ca., bitte verifizieren) |
|---|---|
| Chronos-2 | Oktober 2025 |
| TimesFM-2.5 | September 2025 |
| Moirai-2.0 | August 2025 |
| Toto 2.0 | **offen** |

Deshalb liegen **val und test nach dem 31.10.2025**. Der Preis: val (4 Monate) und test (6 Monate) sind kurz. In der train-Periode können die Foundation Models dagegen zu gut aussehen. Ein Gate könnte dort lernen, ihnen zu sehr zu vertrauen. Genau das sieht man dann auf val.

Im ersten Notebook gelten noch die alten Grenzen (train ≤ 2023, val = 2024). Für die Kombinationen hier zählen nur die Grenzen unten.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from scipy.stats import norm

DATA_DIR, RESULTS_DIR = Path("data"), Path("results")

QUANTILES = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
             0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
QCOLS = [f"q{q:.2f}" for q in QUANTILES]
_LV = np.asarray(QUANTILES)

POOL = ["Chronos-2", "HAR", "GARCH"]   # Kandidaten für die Kombination
BASELINE = "RW"                        # nur Referenz für relative Metriken

# ---- Splits (nach Zieldatum) ----
TRAIN_END = "2025-10-31"   # nach dem Release des jüngsten Experten
VAL_END = "2026-02-28"
ALLOW_TEST = False         # erst ganz am Ende auf True setzen!

# ---- Kombination ----
ALPHA = 1.0                # Gewicht des Pinball-Terms im Trainingsziel
PERF_HALFLIFE = 10         # Halbwertszeit (Ursprünge) für die jüngsten Fehler der Experten
LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]   # L2-Regularisierung der Gates, Auswahl auf val
MLP_HIDDEN = 16

torch.set_default_dtype(torch.float64)

In [ ]:
def qlike(y, f):
    r = y / f
    return r - np.log(r) - 1


def pinball(y, Q):
    d = y[:, None] - Q
    return np.mean(np.maximum(_LV * d, (_LV - 1) * d), axis=1)


def assign_split(target_dates):
    td = pd.to_datetime(target_dates)
    return np.where(td <= pd.Timestamp(TRAIN_END), "train",
                    np.where(td <= pd.Timestamp(VAL_END), "val", "test"))


def select_split(df, split):
    if split == "test" and not ALLOW_TEST:
        raise RuntimeError("Die Testperiode ist gesperrt. Erst für die finale Evaluation öffnen (ALLOW_TEST = True).")
    return df[df["split"] == split]


fc = pd.concat([pd.read_parquet(p) for p in sorted(RESULTS_DIR.glob("*__*.parquet"))], ignore_index=True)
fc = fc[fc["model"].isin(POOL + [BASELINE])].copy()
fc["split"] = assign_split(fc["target_date"])          # neue Grenzen überschreiben die gespeicherten
fc["qlike"] = qlike(fc["y"].to_numpy(), fc["mean"].to_numpy())

display(fc[fc["model"] == POOL[0]].groupby(["h", "split"])
          .agg(n=("y", "size"), von=("target_date", "min"), bis=("target_date", "max")))

## 1. Features für das Gating

Alle Features sind am Ursprung $t$ bekannt, es gibt also kein Leakage.

- **Marktzustand** (aus der täglichen RV): Abweichung der log-RV vom 22-Tage-Mittel, Trend (5-Tage vs. 22-Tage), Niveau, Vola der Vola, standardisierte Rendite des letzten Tages und der letzten 5 Tage.
- **Uneinigkeit:** $\log\hat\mu_m - \overline{\log\hat\mu}$ pro Experte. Wie stark weicht ein Modell vom Rest des Pools ab?
- **Jüngste Fehler:** exponentiell gewichteter QLIKE jedes Experten, relativ zum Pool. Für Horizont $h$ zählen nur Forecasts, deren Ziel bis $t$ bekannt ist (Verschiebung um $h$ Ursprünge).

Die Features werden mit Mittelwert und Streuung aus `train` standardisiert. Fehlende Werte in der Aufwärmphase werden auf 0 gesetzt, also auf den train-Mittelwert.

In [ ]:
def daily_features(sid):
    d = pd.read_parquet(DATA_DIR / f"{sid}_daily.parquet")
    rv, ret = d["rv"], d["ret"]
    lrv, rv22 = np.log(rv), rv.rolling(22).mean()
    f = pd.DataFrame({
        "logrv_dev": lrv - np.log(rv22),
        "trend": np.log(rv.rolling(5).mean()) - np.log(rv22),
        "level": np.log(rv22),
        "volvol": lrv.rolling(22).std(),
        "ret1": ret / np.sqrt(rv22),
        "ret5": ret.rolling(5).sum() / np.sqrt(5 * rv22),
        "scale": rv22,                       # Normierung des Pinball-Loss, kein Feature
    })
    f.index = pd.MultiIndex.from_arrays([np.repeat(sid, len(f)), f.index], names=["sid", "origin"])
    return f


def recent_loss(x):
    h = x.name[1]                            # Gruppenschlüssel (sid, h, model)
    return x.shift(h).ewm(halflife=PERF_HALFLIFE, min_periods=5).mean()


fc = fc.sort_values(["sid", "h", "model", "origin"])
fc["recent"] = fc.groupby(["sid", "h", "model"], group_keys=False)["qlike"].apply(recent_loss)
dfeat = pd.concat([daily_features(sid) for sid in fc["sid"].unique()])
MARKET_FEATS = [c for c in dfeat.columns if c != "scale"]

In [ ]:
def build_panel(h, models=POOL):
    # Ein Horizont als Arrays: mean (n, M), Q (n, M, nQ), Features X (n, k), Metadaten
    sub = fc[(fc["h"] == h) & fc["model"].isin(models)]
    wide = sub.set_index(["sid", "origin", "model"])[["y", "target_date", "mean", "recent"] + QCOLS].unstack("model")
    mean = wide["mean"][models].to_numpy()
    Q = np.stack([wide[c][models].to_numpy() for c in QCOLS], axis=-1)
    ok = np.isfinite(mean).all(1) & np.isfinite(Q).all((1, 2))
    wide, mean, Q = wide[ok], mean[ok], Q[ok]

    meta = pd.DataFrame({"y": wide["y"][models[0]].to_numpy(),
                         "target_date": wide["target_date"][models[0]].to_numpy()}, index=wide.index)
    meta["h"] = h
    meta["split"] = assign_split(meta["target_date"])
    meta = meta.join(dfeat)

    logm = np.log(mean)
    X = meta[MARKET_FEATS].copy()
    for j, m in enumerate(models):
        X[f"disagree_{m}"] = logm[:, j] - logm.mean(1)
    lrec = np.log(wide["recent"][models].to_numpy())
    for j, m in enumerate(models):
        X[f"recent_{m}"] = lrec[:, j] - lrec.mean(1)

    tr = (meta["split"] == "train").to_numpy()
    mu, sd = X[tr].mean(), X[tr].std().replace(0, 1)
    X = ((X - mu) / sd).fillna(0.0)
    return {"h": h, "models": models, "meta": meta.reset_index(), "mean": mean, "Q": Q,
            "X": X.reset_index(drop=True), "scale": meta["scale"].to_numpy()}


panels = {h: build_panel(h) for h in sorted(fc["h"].unique())}
FEATS = list(panels[1]["X"].columns)
print(f"{len(FEATS)} Features:", FEATS)
for h, P in panels.items():
    print(f"h={h}:", P["meta"]["split"].value_counts().to_dict())

## 2. Gate und Training

Statische Gewichte sind ein Gate ohne Features: Es bleibt nur der Bias $b$. Alle gelernten Gewichte starten beim Gleichgewicht ($b = 0$, letzte Schicht $= 0$). Optimiert wird full-batch mit L-BFGS. Die L2-Strafe wirkt nur auf die Feature-Gewichte. Ohne Features bleibt das Gate beim Bias, mit hohem $\lambda$ also bei statischen Gewichten.

In [ ]:
class Gate(nn.Module):
    def __init__(self, n_feat, n_exp, hidden=0):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(n_exp))
        if n_feat == 0:
            self.net = None
        elif hidden == 0:
            self.net = nn.Linear(n_feat, n_exp, bias=False)
        else:
            self.net = nn.Sequential(nn.Linear(n_feat, hidden), nn.Tanh(), nn.Linear(hidden, n_exp, bias=False))
        if self.net is not None:
            nn.init.zeros_((self.net if hidden == 0 else self.net[-1]).weight)

    def forward(self, X):
        z = self.logits.expand(len(X), -1)
        if self.net is not None:
            z = z + self.net(X)
        return torch.softmax(z, dim=-1)

    def penalty(self):
        return sum((p ** 2).sum() for p in self.net.parameters()) if self.net is not None else torch.tensor(0.0)


def _tensors(P, rows, feats):
    T = lambda a: torch.as_tensor(np.asarray(a)[rows])
    return (T(P["X"][feats].to_numpy()) if feats else torch.zeros(int(rows.sum()), 0),
            T(P["mean"]), T(P["Q"]), T(P["meta"]["y"].to_numpy()), T(P["scale"]))


def comb_loss(w, mean, Q, y, s, alpha=ALPHA):
    f = (w * mean).sum(1)
    Qc = torch.einsum("nm,nmq->nq", w, Q)
    r = y / f
    d = y[:, None] - Qc
    lv = torch.as_tensor(_LV)
    pin = torch.maximum(lv * d, (lv - 1) * d).mean(1)
    return (r - torch.log(r) - 1).mean() + alpha * (pin / s).mean()


def fit_gate(P, feats, hidden=0, lam=0.0, seed=0):
    torch.manual_seed(seed)
    rows = (P["meta"]["split"] == "train").to_numpy()
    X, mean, Q, y, s = _tensors(P, rows, feats)
    gate = Gate(len(feats), len(P["models"]), hidden)
    opt = torch.optim.LBFGS(gate.parameters(), lr=1, max_iter=500, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = comb_loss(gate(X), mean, Q, y, s) + lam * gate.penalty()
        loss.backward()
        return loss

    opt.step(closure)
    return gate


def gate_weights(gate, P, feats):
    rows = np.ones(len(P["meta"]), bool)
    with torch.no_grad():
        return gate(_tensors(P, rows, feats)[0]).numpy()


def split_loss(P, w, split):
    rows = (P["meta"]["split"] == split).to_numpy()
    _, mean, Q, y, s = _tensors(P, rows, [])
    return float(comb_loss(torch.as_tensor(w[rows]), mean, Q, y, s))

In [ ]:
weights, tuning = {}, []
for h, P in panels.items():
    n, M = len(P["meta"]), len(P["models"])
    W = {"Gleichgewicht": np.full((n, M), 1 / M)}

    tr = (P["meta"]["split"] == "train").to_numpy()
    best = int(np.argmin(qlike(P["meta"]["y"].to_numpy()[tr, None], P["mean"][tr]).mean(0)))
    W["Auswahl"] = np.eye(M)[np.full(n, best)]

    W["Statisch"] = gate_weights(fit_gate(P, []), P, [])

    for name, hidden in [("Gating linear", 0), ("Gating MLP", MLP_HIDDEN)]:
        best_val = None
        for lam in LAMBDAS:
            w = gate_weights(fit_gate(P, FEATS, hidden, lam), P, FEATS)
            vl = split_loss(P, w, "val")
            tuning.append({"h": h, "Methode": name, "lambda": lam,
                           "Loss train": split_loss(P, w, "train"), "Loss val": vl})
            if best_val is None or vl < best_val:
                best_val, W[name] = vl, w
    weights[h] = W

tuning = pd.DataFrame(tuning)
display(tuning.pivot_table(index=["h", "lambda"], columns="Methode", values=["Loss train", "Loss val"]).round(4))
print("Hinweis: λ wird auf val gewählt, die val-Ergebnisse unten sind für die Gates daher leicht optimistisch.")

## 3. Evaluation auf der Validierungsperiode

Einzelne Experten und Kombinationen werden auf **denselben Zeilen** ausgewertet. Die Experten erscheinen dabei als One-Hot-Gewichte. Die Metriken sind dieselben wie im ersten Notebook, relativ zum Random Walk und über Reihen per geometrischem Mittel aggregiert. **< 1 heißt besser als der Random Walk.**

In [ ]:
def to_long(P, w, name):
    df = P["meta"][["sid", "h", "origin", "target_date", "split", "y"]].copy()
    df["model"] = name
    df["mean"] = (w * P["mean"]).sum(1)
    df[QCOLS] = np.einsum("nm,nmq->nq", w, P["Q"])
    return df


rows = []
for h, P in panels.items():
    M = len(P["models"])
    for j, m in enumerate(P["models"]):
        rows.append(to_long(P, np.eye(M)[np.full(len(P["meta"]), j)], m))
    for name, w in weights[h].items():
        rows.append(to_long(P, w, name))
cand = pd.concat(rows, ignore_index=True)

keys = cand[["sid", "h", "origin"]].drop_duplicates()
rw = fc[fc["model"] == BASELINE].merge(keys, on=["sid", "h", "origin"])
allfc = pd.concat([cand, rw[cand.columns]], ignore_index=True)
allfc["qlike"] = qlike(allfc["y"].to_numpy(), allfc["mean"].to_numpy())
allfc["pinball"] = pinball(allfc["y"].to_numpy(), allfc[QCOLS].to_numpy())
allfc["cov90"] = (allfc["y"] >= allfc["q0.05"]) & (allfc["y"] <= allfc["q0.95"])

ORDER = [BASELINE] + POOL + list(weights[1].keys())
EVAL_SPLIT = "val"
ev = select_split(allfc, EVAL_SPLIT)

agg = (ev.groupby(["sid", "h", "model"])
         .agg(QLIKE=("qlike", "mean"), Pinball=("pinball", "mean"), Cov90=("cov90", "mean"))
         .reset_index())
base = agg.loc[agg["model"] == BASELINE, ["sid", "h", "QLIKE", "Pinball"]]
agg = agg.merge(base, on=["sid", "h"], suffixes=("", "_RW"))
agg["rel_QLIKE"] = agg["QLIKE"] / agg["QLIKE_RW"]
agg["rel_Pinball"] = agg["Pinball"] / agg["Pinball_RW"]

gmean = lambda x: float(np.exp(np.log(x).mean()))
summary = (agg.groupby(["h", "model"])
              .agg(rel_QLIKE=("rel_QLIKE", gmean), rel_Pinball=("rel_Pinball", gmean), Cov90=("Cov90", "mean"))
              .round(3))
summary = summary.reindex(pd.MultiIndex.from_product([sorted(panels), ORDER], names=["h", "model"]))
print(f"{EVAL_SPLIT}: {ev.groupby('model').size().iloc[0]:,} Forecasts pro Modell")
display(summary)
display(agg.pivot_table(index=["sid", "h"], columns="model", values="rel_QLIKE")[ORDER].round(3))

In [ ]:
def dm_test(d, h):
    # Diebold-Mariano auf Loss-Differenzen d, HAC-Varianz nach Newey-West
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    T, u = len(d), d - d.mean()
    lag = max(h - 1, 1)
    var = u @ u / T + 2 * sum((1 - k / (lag + 1)) * (u[k:] @ u[:-k]) / T for k in range(1, lag + 1))
    stat = d.mean() / np.sqrt(var / T)
    return stat, 2 * (1 - norm.cdf(abs(stat)))


PAIRS = [("Gating linear", "Chronos-2"), ("Gating linear", "Statisch"),
         ("Statisch", "Chronos-2"), ("Gleichgewicht", "Chronos-2")]
L_all = ev.pivot_table(index=["sid", "h", "origin"], columns="model", values="qlike")
rows = []
for (sid, h), g in L_all.groupby(level=["sid", "h"]):
    for a, b in PAIRS:
        stat, p = dm_test(g[a] - g[b], h)
        rows.append({"sid": sid, "h": h, "Vergleich": f"{a} − {b}", "DM": round(stat, 2), "p": round(p, 3)})
display(pd.DataFrame(rows).pivot_table(index=["sid", "h"], columns="Vergleich", values=["DM", "p"]))
print("DM < 0: das erste Modell hat den kleineren QLIKE")

## 4. Was lernen die Kombinationen?

In [ ]:
static = pd.DataFrame({f"h={h}": weights[h]["Statisch"][0] for h in panels}, index=POOL)
print("Statische Gewichte:")
display(static.round(3))

gw = []
for h, P in panels.items():
    df = P["meta"][["sid", "split"]].copy()
    df[POOL] = weights[h]["Gating linear"]
    df["h"] = h
    gw.append(df[df["split"] != "test"])
display(pd.concat(gw).groupby(["h", "split", "sid"])[POOL].mean().round(3))

In [ ]:
# Gate-Gewichte über die Zeit (h=1, gleitendes 30-Tage-Mittel), Linie = Beginn val
P = panels[1]
df = P["meta"][["sid", "origin", "split"]].copy()
df[POOL] = weights[1]["Gating linear"]
df = df[df["split"] != "test"]

sids = df["sid"].unique()
fig, axes = plt.subplots(len(sids), 1, figsize=(11, 2.6 * len(sids)), sharex=True, squeeze=False)
for ax, sid in zip(axes[:, 0], sids):
    d = df[df["sid"] == sid].set_index("origin")
    for m in POOL:
        ax.plot(d[m].rolling(30, min_periods=10).mean(), label=m, lw=1)
    ax.axvline(pd.Timestamp(TRAIN_END), color="grey", ls="--", lw=0.8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{sid}: Gating-Gewichte (linear, h=1)", fontsize=10)
axes[0, 0].legend(ncol=len(POOL), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Welche Features treiben das lineare Gate? (Gewichte auf standardisierten Features, h=1)
gate = fit_gate(panels[1], FEATS, 0, tuning.query("h == 1 and Methode == 'Gating linear'")
                .sort_values("Loss val")["lambda"].iloc[0])
coef = pd.DataFrame(gate.net.weight.detach().numpy().T, index=FEATS, columns=POOL)
display(coef.round(3))

## 5. Nächste Schritte

1. **Weitere Experten:** TimesFM-2.5 als `Expert`-Klasse im ersten Notebook ergänzen, `run_expert` laufen lassen, hier in `POOL` aufnehmen. Danach Toto und Moirai (Moirai erst nach dem Fix aus dem Resource-Check).
2. **Robustheit:** längere val-Periode testen (z.B. `TRAIN_END` früher setzen) und schauen, wie stabil Ranking und Gewichte sind.
3. **Gate verfeinern:** Sparse Top-k-Gating nach Shazeer, ein Gate über alle Horizonte mit $h$ als Feature, Gewichte pro Quantil.
4. **Weitere Domänen:** Aktien und Indizes, Standard-Benchmarks.
5. **Testperiode** erst ganz am Ende öffnen (`ALLOW_TEST = True`).